# TMDB 영화 데이터셋 전처리 실습

> 🎬 **TMDB 5000 영화 데이터셋**을 활용하여 데이터 전처리부터 파생 변수 생성, EDA까지 실습합니다.

## 🎯 학습 목표
1. 두 개의 CSV 파일을 로드하고 `merge`로 결합할 수 있다.
2. `literal_eval`을 이용해 문자열로 저장된 JSON 컬럼을 파이썬 객체로 변환할 수 있다.
3. `apply` + `lambda` 패턴으로 파생 변수를 생성할 수 있다.
4. 결측값·이상값을 탐지하고 제거할 수 있다.
5. `groupby`, `explode`로 집계 점수를 계산하고 병합할 수 있다.
6. Min-Max Scaling으로 정규화된 점수를 생성할 수 있다.
7. 전처리된 데이터셋을 CSV로 저장하고 기본 EDA를 수행할 수 있다.

## 📁 필요 파일
| 파일명 | 설명 |
|---|---|
| `tmdb_5000_movies.csv` | 영화 기본 정보 (장르, 예산, 수익 등) |
| `tmdb_5000_credits.csv` | 출연진 & 제작진 정보 |

## 📋 전체 흐름
| Step | 내용 |
|---|---|
| Step 1 | 라이브러리 임포트 & 파일 경로 설정 |
| Step 2 | 원본 데이터 로드 & 병합 |
| Step 3 | 전처리 보조 함수 정의 |
| Step 4 | 파생 변수 생성 |
| Step 5 | 결측값 & 이상값 제거 |
| Step 6 | 감독·배우 인지도 점수 계산 |
| Step 7 | 마케팅 점수 생성 |
| Step 8 | 최종 컬럼 선택 & CSV 저장 |
| Step 9 | 데이터 탐색 (EDA) |


## Step 1: 라이브러리 임포트 & 파일 경로 설정

### 📌 사용 라이브러리
| 라이브러리 | 용도 |
|---|---|
| `pandas` | 데이터프레임 처리 |
| `numpy` | 수치 연산, NaN 처리 |
| `re` | 정규표현식 (텍스트 패턴 찾기) — 이후 텍스트 전처리 확장 시 사용 |
| `literal_eval` | 문자열로 저장된 파이썬 객체(리스트, 딕셔너리)를 안전하게 변환 |


In [1]:
# ✅ 필요한 라이브러리 임포트
import pandas as pd
import numpy as np
import re                       # 정규표현식 (텍스트 패턴 찾기)
from ast import literal_eval    # 문자열 → 파이썬 리스트/딕셔너리 안전 변환

# 파일 경로 정의
movies_path  = 'tmdb_5000_movies.csv'
credits_path = 'tmdb_5000_credits.csv'
output_path  = 'movie_box_office_prediction_tmdb_like.csv'

print('✅ 라이브러리 로드 완료')


✅ 라이브러리 로드 완료


## Step 2: 원본 데이터 로드 & 병합

### 📌 `merge` 방식 비교
| 방식 | 의미 |
|---|---|
| `how='inner'` | 양쪽 모두 있는 행만 남김 |
| `how='left'` | 왼쪽(movies) 전체 유지 + 오른쪽(credits) 매칭되는 것 추가 ✅ |
| `how='outer'` | 양쪽 모두 유지 |

- `on='title'`: 영화 제목이 완전히 일치하는 경우만 병합합니다. 오타가 있으면 NaN이 발생합니다.
- 병합 후 `shape`를 확인하여 행 수 변화를 파악하세요.


In [2]:
# 두 CSV를 로드하고 title 기준으로 left join 병합
movies  = pd.read_csv(movies_path)
credits = pd.read_csv(credits_path)

print(f'movies shape:  {movies.shape}')
print(f'credits shape: {credits.shape}')

df = movies.merge(credits, on='title', how='left')

print(f'\n병합 후 shape: {df.shape}')
df.head(3)


movies shape:  (4803, 20)
credits shape: (4803, 4)

병합 후 shape: (4809, 23)


,budget,genres,homepage,id,keywords,original_language,original_title,overview,popularity,production_companies,...,runtime,spoken_languages,status,tagline,title,vote_average,vote_count,movie_id,cast,crew
0,237000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.avatarmovie.com/,19995,"[{""id"": 1463, ""name"": ""culture clash""}, {""id"":...",en,Avatar,"In the 22nd century, a paraplegic Marine is di...",150.437577,"[{""name"": ""Ingenious Film Partners"", ""id"": 289...",...,162.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}, {""iso...",Released,Enter the World of Pandora.,Avatar,7.2,11800,19995,"[{""cast_id"": 242, ""character"": ""Jake Sully"", ""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""de..."
1,300000000,"[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""...",http://disney.go.com/disneypictures/pirates/,285,"[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""na...",en,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, ha...",139.082615,"[{""name"": ""Walt Disney Pictures"", ""id"": 2}, {""...",...,169.0,"[{""iso_639_1"": ""en"", ""name"": ""English""}]",Released,"At the end of the world, the adventure begins.",Pirates of the Caribbean: At World's End,6.9,4500,285,"[{""cast_id"": 4, ""character"": ""Captain Jack Spa...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""de..."
2,245000000,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""nam...",http://www.sonypictures.com/movies/spectre/,206647,"[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name...",en,Spectre,A cryptic message from Bond’s past sends him o...,107.376788,"[{""name"": ""Columbia Pictures"", ""id"": 5}, {""nam...",...,148.0,"[{""iso_639_1"": ""fr"", ""name"": ""Fran\u00e7ais""},...",Released,A Plan No One Escapes,Spectre,6.3,4466,206647,"[{""cast_id"": 1, ""character"": ""James Bond"", ""cr...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""de..."


## Step 3: 전처리 보조 함수 정의

### 📌 `literal_eval`이 필요한 이유

TMDB CSV의 `genres`, `cast` 등의 컬럼은 **문자열처럼 보이지만 실제로는 리스트/딕셔너리 구조**입니다.

```python
# CSV에 저장된 실제 값 (문자열 타입)
raw = '[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}]'
type(raw)      # → str  ← 문자열!

# literal_eval 적용 후
parsed = literal_eval(raw)
type(parsed)   # → list ← 리스트로 변환!
parsed[0]      # → {'id': 28, 'name': 'Action'}
```

> **`literal_eval` vs `eval()`**  
> `eval()`은 임의의 코드를 실행할 수 있어 보안상 위험합니다.  
> `literal_eval`은 파이썬 리터럴(숫자, 문자열, 리스트, 딕셔너리 등)만 처리하므로 **안전**합니다.
>
> **`literal_eval` vs `json.loads`**  
> `json.loads`는 JSON 표준만 처리하지만, `literal_eval`은 `True/False/None` 같은 파이썬 리터럴도 처리 가능합니다.


In [3]:
# safe_literal_eval — 문자열 → 파이썬 객체 변환
# NaN이거나 변환에 실패하면 빈 리스트 [] 를 반환합니다
def safe_literal_eval(x):
    """
    CSV 안의 텍스트가 '[{"id": 1, "name": "Action"}]' 같은 문자열일 때,
    이를 실제 파이썬 리스트 객체로 변환해주는 안전한 함수
    """
    if pd.isna(x):
        return []
    try:
        return literal_eval(x)
    except:
        return []

# 5개 컬럼에 일괄 적용
for col in ['genres', 'keywords', 'production_companies', 'cast', 'crew']:
    if col in df.columns:
        df[col] = df[col].apply(safe_literal_eval)

# 변환 전후 타입 확인
print('genres 컬럼 첫 번째 값 타입:', type(df['genres'].iloc[0]))
print('genres 첫 번째 값:', df['genres'].iloc[0][:2], '...')


genres 컬럼 첫 번째 값 타입: <class 'list'>
genres 첫 번째 값: [{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}] ...


### 📌 `extract_names` — 딕셔너리 리스트에서 `'name'` 값만 추출

```python
# 예시 입력
sample = [{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}]

extract_names(sample)          # → ['Action', 'Adventure']
extract_names(sample, top_n=1) # → ['Action']
```

- 리스트 컴프리헨션으로 `'name'` 키의 값만 뽑아냅니다.
- `top_n`을 지정하면 슬라이싱으로 상위 N개만 반환합니다.


In [4]:
# extract_names — 딕셔너리 리스트에서 'name' 값만 추출
def extract_names(items, top_n=None):
    """
    리스트 안에 딕셔너리 구조(예: [{'name': '액션'}, {'name': '스릴러'}])에서
    'name'에 해당하는 값들만 뽑아내는 함수
    top_n을 지정하면 상위 N개만 추출
    """
    if not isinstance(items, list):
        return []
    names = [
        item.get('name', '')
        for item in items
        if isinstance(item, dict) and item.get('name')
    ]
    return names[:top_n] if top_n else names

# 테스트
sample = [{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}, {'id': 14, 'name': 'Fantasy'}]
print('전체 추출:', extract_names(sample))
print('상위 2개:', extract_names(sample, top_n=2))


전체 추출: ['Action', 'Adventure', 'Fantasy']
상위 2개: ['Action', 'Adventure']


### 📌 `extract_director` — 제작진 리스트에서 감독 이름 추출

```python
# crew 리스트 구조 예시
crew = [
    {'job': 'Producer',   'name': 'John Smith'},
    {'job': 'Director',   'name': 'James Cameron'},  # ← 이 사람만 추출
    {'job': 'Screenplay', 'name': 'Jane Doe'}
]
extract_director(crew)  # → 'James Cameron'
```

- `job == 'Director'`인 첫 번째 사람의 이름을 반환합니다.
- 감독이 없으면 `np.nan`을 반환합니다.


In [5]:
# extract_director — 제작진 리스트에서 감독 이름 추출
def extract_director(crew_list):
    """
    제작진(crew) 리스트를 돌면서 job이 'Director'인 사람의 이름만 반환
    감독이 없으면 np.nan 반환
    """
    if not isinstance(crew_list, list):
        return np.nan
    for person in crew_list:
        if isinstance(person, dict) and person.get('job') == 'Director':
            return person.get('name')
    return np.nan

# 테스트
sample_crew = [
    {'job': 'Producer',   'name': 'John Smith'},
    {'job': 'Director',   'name': 'James Cameron'},
    {'job': 'Screenplay', 'name': 'Jane Doe'}
]
print('감독 추출 결과:', extract_director(sample_crew))


감독 추출 결과: James Cameron


## Step 4: 파생 변수 생성

### 📌 `apply` + `lambda` 패턴

아래 두 코드는 **완전히 동일**합니다.

```python
# lambda 방식 (간결)
df['genre'] = df['genres'].apply(
    lambda x: extract_names(x, 1)[0] if len(extract_names(x, 1)) > 0 else 'Unknown'
)

# 함수로 풀어쓰면
def get_first_genre(x):
    names = extract_names(x, 1)
    return names[0] if len(names) > 0 else 'Unknown'

df['genre'] = df['genres'].apply(get_first_genre)
```

### 📌 `pd.to_numeric(errors='coerce')` 옵션
| 옵션 | 동작 |
|---|---|
| `errors='raise'` | 변환 실패 시 오류 발생 (기본값) |
| `errors='coerce'` | 변환 불가능한 값 → **NaN**으로 처리 ✅ |
| `errors='ignore'` | 변환 실패 시 원본 값 유지 |

### 📌 생성할 파생 변수 6가지
| 컬럼명 | 내용 |
|---|---|
| `genre` | 첫 번째 장르 (없으면 `'Unknown'`) |
| `director_name` | 감독 이름 |
| `cast_names` | 주연 배우 3명 리스트 |
| `runtime` 등 6개 컬럼 | 수치형으로 강제 변환 |
| `release_month`, `release_year` | 개봉일에서 월/연도 분리 |
| `production_company_size` | 참여 제작사 수 |


In [6]:
# 파생 변수 6개 생성

# (1) 장르: 첫 번째 장르 1개 (없으면 'Unknown')
df['genre'] = df['genres'].apply(
    lambda x: extract_names(x, 1)[0] if len(extract_names(x, 1)) > 0 else 'Unknown'
)

# (2) 감독 이름
df['director_name'] = df['crew'].apply(extract_director)

# (3) 주연 배우 3명 리스트
df['cast_names'] = df['cast'].apply(lambda x: extract_names(x, 3))

# (4) 수치형 컬럼 강제 변환 (문자열/오류 → NaN)
for col in ['runtime', 'budget', 'revenue', 'vote_average', 'vote_count', 'popularity']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# (5) 개봉일 → 월/연도 분리
df['release_date']  = pd.to_datetime(df['release_date'], errors='coerce')
df['release_month'] = df['release_date'].dt.month
df['release_year']  = df['release_date'].dt.year

# (6) 참여 제작사 수
df['production_company_size'] = df['production_companies'].apply(
    lambda x: len(extract_names(x))
)

print('파생 변수 생성 완료')
print(df[['title', 'genre', 'director_name', 'cast_names', 'release_year', 'production_company_size']].head(3))


파생 변수 생성 완료
                                      title      genre   director_name  \
0                                    Avatar     Action   James Cameron   
1  Pirates of the Caribbean: At World's End  Adventure  Gore Verbinski   
2                                   Spectre     Action      Sam Mendes   

                                         cast_names  release_year  \
0  [Sam Worthington, Zoe Saldana, Sigourney Weaver]        2009.0   
1     [Johnny Depp, Orlando Bloom, Keira Knightley]        2007.0   
2      [Daniel Craig, Christoph Waltz, Léa Seydoux]        2015.0   

   production_company_size  
0                        4  
1                        3  
2                        3  


## Step 5: 결측값 & 이상값 제거

### 📌 왜 `budget = 0`인 행을 제거하나요?

TMDB 데이터에서 `0`은 대부분 **'데이터 미입력'** 을 의미합니다.  
실제 독립 저예산 영화는 0이 아닌 소액으로 등록되어 있습니다.  
이처럼 **도메인 지식**이 전처리 기준을 결정한다는 점을 기억하세요.

### 📌 비교 전 `.fillna(0)` 를 쓰는 이유
```python
# NaN > 0 은 Python에서 False를 반환 → NaN 행도 제거됨!
# 비교 전에 NaN을 0으로 채워야 의도한 대로 동작합니다.
df['budget'].fillna(0) > 0
```


In [7]:
# 비정상 데이터(budget / revenue / runtime = 0 or NaN) 제거
before = len(df)

df = df[
    (df['budget'].fillna(0)  > 0) &
    (df['revenue'].fillna(0) > 0) &
    (df['runtime'].fillna(0) > 0)
].copy()

after = len(df)
print(f'제거 전: {before}행 → 제거 후: {after}행 (제거: {before - after}행)')
print(f'\n결측 요약:\n{df[["budget","revenue","runtime","director_name"]].isnull().sum()}')


제거 전: 4809행 → 제거 후: 3232행 (제거: 1577행)

결측 요약:
budget           0
revenue          0
runtime          0
director_name    2
dtype: int64


## Step 6: 감독 & 배우 인지도(Popularity) 점수 계산

### 📌 `groupby` → `merge` 패턴

같은 감독이 여러 영화에 등장하므로, 감독별 평균 인지도를 계산한 뒤 원본에 병합합니다.

```
[df 원본]
title       director_name      popularity
Avatar      James Cameron      150.4
Titanic     James Cameron      200.1
Inception   Christopher N.      80.2

[groupby 결과]
director_name          director_popularity_raw
James Cameron          175.25  ← (150.4 + 200.1) / 2
Christopher Nolan       80.2

[merge 후]
title       director_name      director_popularity_raw
Avatar      James Cameron      175.25
Titanic     James Cameron      175.25  ← 동일 감독 → 동일 점수
Inception   Christopher N.      80.2
```


In [8]:
# 감독 인지도 점수 계산 & 병합
director_pop = (
    df.groupby('director_name')['popularity']
    .mean()
    .reset_index()
    .rename(columns={'popularity': 'director_popularity_raw'})
)

df = df.merge(director_pop, on='director_name', how='left')

print('감독 인지도 Top 5:')
print(director_pop.sort_values('director_popularity_raw', ascending=False).head(5))


감독 인지도 Top 5:
        director_name  director_popularity_raw
781        Kyle Balda               875.581305
1341       Tim Miller               514.569956
546        James Gunn               246.651596
227   Colin Trevorrow               221.947277
246   Damien Chazelle               192.528841


### 📌 `explode()` 패턴 — 1:N 관계 데이터 처리

`cast_names` 컬럼에는 배우 **리스트**가 들어 있습니다.  
`explode()`를 사용하면 리스트를 **행 단위로 펼쳐** 배우별로 처리할 수 있습니다.

```
[explode 전]
title     cast_names
Avatar    ['Sam W.', 'Zoe S.', 'Sigourney W.']
Titanic   ['Leo D.', 'Kate W.']

[explode('cast_names') 후]
title     cast_name
Avatar    Sam W.
Avatar    Zoe S.
Avatar    Sigourney W.
Titanic   Leo D.
Titanic   Kate W.
```

> SQL의 `UNNEST` / `LATERAL JOIN`과 동일한 개념입니다.


In [9]:
# 배우 인지도 점수 계산 & 병합

# (1) cast_names 리스트를 행으로 펼치기
cast_exploded = df[['title', 'popularity', 'cast_names']].explode('cast_names')
cast_exploded = cast_exploded.rename(columns={'cast_names': 'cast_name'})
cast_exploded = cast_exploded.dropna(subset=['cast_name'])

# (2) 배우별 평균 인지도 계산
actor_pop = (
    cast_exploded.groupby('cast_name')['popularity']
    .mean()
    .reset_index()
    .rename(columns={'popularity': 'actor_popularity_raw'})
)
cast_exploded = cast_exploded.merge(actor_pop, on='cast_name', how='left')

# (3) 영화별 평균 배우 인지도
movie_cast_pop = (
    cast_exploded.groupby('title')['actor_popularity_raw']
    .mean()
    .reset_index()
    .rename(columns={'actor_popularity_raw': 'cast_popularity_raw'})
)

df = df.merge(movie_cast_pop, on='title', how='left')

print('배우 인지도 Top 5:')
print(actor_pop.sort_values('actor_popularity_raw', ascending=False).head(5))


배우 인지도 Top 5:
            cast_name  actor_popularity_raw
2508  Morena Baccarin            514.569956
777     Dave Bautista            481.098624
1753         Jon Hamm            446.446869
948         Ed Skrein            269.786336
3094      Scott Adsit            203.734590


## Step 7: 마케팅 점수(marketing_score) 생성

### 📌 Min-Max Scaling

서로 다른 단위와 범위를 가진 값들을 **0 ~ 100 범위**로 통일하는 정규화 방법입니다.

$$
\text{score} = \frac{x - x_{\min}}{x_{\max} - x_{\min}} \times 100
$$

**예시**: popularity = [10, 50, 100]
- $x_{\min} = 10$, $x_{\max} = 100$
- $10 \rightarrow 0$ 점, $50 \rightarrow 44.4$ 점, $100 \rightarrow 100$ 점

### 📌 `log1p(x)`를 쓰는 이유
`budget` 같은 값은 범위가 너무 크기 때문에 (1,000 ~ 300,000,000),  
로그 변환으로 스케일을 줄여 블록버스터가 점수를 독식하지 않도록 합니다.  
`log1p(x) = log(1 + x)` → `x = 0`일 때 `log(0)` 오류를 방지합니다.

### 📌 마케팅 점수 가중치
| 요소 | 가중치 | 이유 |
|---|---|---|
| `popularity` | 50% | 현재 인지도 — 가장 중요 |
| `vote_count` | 30% | 투표 참여 수 → 화제성 |
| `budget` | 20% | 마케팅 투자 규모 |


In [10]:
# Min-Max Scaling 함수
def minmax_series(s):
    """시리즈를 0~100 범위로 Min-Max 정규화"""
    s = s.fillna(s.median())
    min_v = s.min()
    max_v = s.max()
    if max_v == min_v:
        return pd.Series([50.0] * len(s), index=s.index)
    return (s - min_v) / (max_v - min_v) * 100


df['popularity_scaled']  = minmax_series(df['popularity'])
df['vote_count_scaled']  = minmax_series(np.log1p(df['vote_count'].fillna(0)))
df['budget_scaled']      = minmax_series(np.log1p(df['budget'].fillna(0)))

# 가중 합산: popularity 50% + vote_count 30% + budget 20%
df['marketing_score'] = (
    df['popularity_scaled'] * 0.5 +
    df['vote_count_scaled'] * 0.3 +
    df['budget_scaled']     * 0.2
)

print('marketing_score 통계:')
print(df['marketing_score'].describe().round(2))


marketing_score 통계:
count    3232.00
mean       37.56
std         6.99
min         7.58
25%        33.33
50%        37.65
75%        41.80
max        94.82
Name: marketing_score, dtype: float64


## Step 7-B: `hit_10M` — 천만 관객 돌파 여부 파생 변수 생성

### 📌 타깃 변수(Label) 개념

- TMDB의 `revenue`는 **달러(\$) 기준**입니다.
- 한국 영화 기준 '천만 관객 × 평균 티켓가 \$10 ≒ \$100,000,000'을 임계값으로 사용합니다.
- 이 컬럼은 이후 **머신러닝 이진 분류(Binary Classification)** 의 타깃 변수로 사용됩니다.

```
revenue ≥ 100,000,000 → hit_10M = 1  (천만 돌파 ✅)
revenue < 100,000,000 → hit_10M = 0  (미달    ❌)
```

> **임계값이 적절한지 확인하기**: `df['revenue'].describe()`와  
> `df['revenue'].hist()`로 분포를 직접 확인해 보세요.


In [11]:
# hit_10M — 천만 관객 돌파 여부 파생 변수 생성
# TMDB revenue는 달러($) 기준: 1,000만 관객 × 평균 티켓가 $10 = $100,000,000
df['hit_10M'] = (df['revenue'] >= 100_000_000).astype(int)

print('hit_10M 빈도수:')
print(df['hit_10M'].value_counts().sort_index())
print(f'\n천만 관객 돌파율: {(df["hit_10M"] == 1).mean() * 100:.2f}%')


hit_10M 빈도수:
hit_10M
0    2107
1    1125
Name: count, dtype: int64

천만 관객 돌파율: 34.81%


## Step 8: 최종 컬럼 선택 & CSV 저장

### 📌 `encoding='utf-8-sig'` 를 사용하는 이유
| 인코딩 | 특징 |
|---|---|
| `utf-8` | 기본 UTF-8. 엑셀에서 열면 한글이 깨질 수 있음 |
| `utf-8-sig` | BOM(Byte Order Mark) 포함. **엑셀이 자동으로 UTF-8로 인식** ✅ |

비개발자와 데이터를 공유할 때 반드시 확인해야 할 포인트입니다.


In [12]:
# ✅ 최종 컬럼 정의
final_cols = [
    'title', 'genre', 'director_name', 'cast_names',
    'budget', 'revenue', 'runtime',
    'vote_average', 'vote_count', 'popularity',
    'release_month', 'release_year', 'production_company_size',
    'director_popularity_raw', 'cast_popularity_raw',
    'marketing_score',
    'hit_10M'   # 천만 관객 돌파 여부 — 머신러닝 타깃 변수
]
final_cols_exist = [c for c in final_cols if c in df.columns]
df_final = df[final_cols_exist].copy()
print(f'최종 데이터셋: {df_final.shape[0]}행 × {df_final.shape[1]}열')
df_final.head()


최종 데이터셋: 3232행 × 17열


,title,genre,director_name,cast_names,budget,revenue,runtime,vote_average,vote_count,popularity,release_month,release_year,production_company_size,director_popularity_raw,cast_popularity_raw,marketing_score,hit_10M
0,Avatar,Action,James Cameron,"[Sam Worthington, Zoe Saldana, Sigourney Weaver]",237000000,2787965087,162.0,7.2,11800,150.437577,12.0,2009.0,4,79.684543,60.335052,57.612536,1
1,Pirates of the Caribbean: At World's End,Adventure,Gore Verbinski,"[Johnny Depp, Orlando Bloom, Keira Knightley]",300000000,961000000,169.0,6.9,4500,139.082615,5.0,2007.0,3,95.845222,71.802547,54.176833,1
2,Spectre,Action,Sam Mendes,"[Daniel Craig, Christoph Waltz, Léa Seydoux]",245000000,880674609,148.0,6.3,4466,107.376788,10.0,2015.0,3,56.151756,71.290412,52.129877,1
3,The Dark Knight Rises,Action,Christopher Nolan,"[Christian Bale, Michael Caine, Gary Oldman]",250000000,1084939099,165.0,7.6,9106,112.312950,7.0,2012.0,4,185.373245,49.986075,54.675561,1
4,John Carter,Action,Andrew Stanton,"[Taylor Kitsch, Lynn Collins, Samantha Morton]",260000000,284139100,132.0,6.1,2124,43.926995,3.0,2012.0,1,70.839325,39.044125,46.229845,1


In [13]:
# cast_names 리스트를 문자열로 변환 후 CSV 저장
df_save = df_final.copy()
df_save['cast_names'] = df_save['cast_names'].apply(
    lambda x: ', '.join(x) if isinstance(x, list) else x
)

df_save.to_csv(output_path, index=False, encoding='utf-8-sig')

print(f'✅ 저장 완료: {output_path}')
print(f'   최종 shape: {df_save.shape}')
print(f'\n컬럼 목록:')
for c in df_save.columns:
    print(f'  - {c}: {df_save[c].dtype} (결측: {df_save[c].isnull().sum()}개)')


✅ 저장 완료: movie_box_office_prediction_tmdb_like.csv
   최종 shape: (3232, 17)

컬럼 목록:
  - title: object (결측: 0개)
  - genre: object (결측: 0개)
  - director_name: object (결측: 2개)
  - cast_names: object (결측: 0개)
  - budget: int64 (결측: 0개)
  - revenue: int64 (결측: 0개)
  - runtime: float64 (결측: 0개)
  - vote_average: float64 (결측: 0개)
  - vote_count: int64 (결측: 0개)
  - popularity: float64 (결측: 0개)
  - release_month: float64 (결측: 0개)
  - release_year: float64 (결측: 0개)
  - production_company_size: int64 (결측: 0개)
  - director_popularity_raw: float64 (결측: 2개)
  - cast_popularity_raw: float64 (결측: 2개)
  - marketing_score: float64 (결측: 0개)
  - hit_10M: int64 (결측: 0개)


## Step 9: 데이터 탐색 (EDA)

### 📌 EDA(Exploratory Data Analysis)란?
데이터의 분포, 패턴, 이상값을 시각화와 통계로 파악하는 탐색 과정입니다.  
모델링 전에 반드시 수행해야 데이터의 특성을 이해하고 올바른 전처리 기준을 잡을 수 있습니다.

### 📌 탐색 시 생각해볼 질문
1. 가장 흥행한 장르는 무엇인가요? 그 이유는?
2. `marketing_score`가 높은데 `revenue`가 낮은 영화가 있나요? 왜 그럴까요?
3. `director_popularity_raw`와 `revenue` 사이에 상관관계가 있을까요?


In [ ]:
# EDA — 기본 통계 확인
df_check = pd.read_csv(output_path)

# Q1. 장르별 영화 수 Top 10
print('=== 장르별 영화 수 Top 10 ===')
print(df_check['genre'].value_counts().head(10))

# Q2. 수익 Top 5 영화
print('\n=== 수익 Top 5 영화 ===')
print(df_check.nlargest(5, 'revenue')[['title', 'revenue', 'budget', 'genre']])

# Q3. marketing_score Top 5 영화
print('\n=== 마케팅 점수 Top 5 영화 ===')
print(df_check.nlargest(5, 'marketing_score')[['title', 'marketing_score', 'director_name']])


## 💡 심화 도전 과제

기본 실습을 완료한 후 아래 도전 과제를 직접 구현해 보세요.

1. `revenue / budget`으로 **ROI(투자수익률)** 컬럼을 추가하고 ROI 상위 10개 영화를 확인하세요.
2. `release_month`별 평균 `revenue`를 **막대그래프**로 시각화하세요.
3. `director_popularity_raw`가 높은 감독과 낮은 감독의 평균 `revenue` 차이를 비교하세요.
4. `vote_average` 기준 8.0 이상 영화만 추출하여 장르 분포를 **파이차트**로 그려보세요.


In [ ]:
# 심화 1: ROI 계산
df_check['ROI'] = df_check['revenue'] / df_check['budget']
print('=== ROI Top 10 ===')
print(df_check.nlargest(10, 'ROI')[['title', 'ROI', 'budget', 'revenue']].round(2))

# 심화 2: 월별 평균 수익
monthly = df_check.groupby('release_month')['revenue'].mean().reset_index()
print('\n=== 월별 평균 수익 (단위: 억) ===')
monthly['revenue_億'] = (monthly['revenue'] / 1e8).round(1)
print(monthly.to_string(index=False))

# 심화 3: 감독 인지도 상위/하위 집단 수익 비교
median_dir_pop = df_check['director_popularity_raw'].median()
high_dir = df_check[df_check['director_popularity_raw'] >= median_dir_pop]['revenue'].mean()
low_dir  = df_check[df_check['director_popularity_raw'] <  median_dir_pop]['revenue'].mean()
print(f'\n=== 감독 인지도 상위 평균 수익: {high_dir/1e8:.1f}억 vs 하위: {low_dir/1e8:.1f}억 ===')


## 📋 학습 체크리스트

실습 후 아래 항목을 스스로 점검해 보세요.

- [ ] `literal_eval` 과 `eval()` 의 차이를 설명할 수 있다.
- [ ] `merge(how='left')` 의 동작 방식을 설명할 수 있다.
- [ ] `explode()` 패턴이 언제 필요한지 설명할 수 있다.
- [ ] Min-Max Scaling 수식을 직접 쓸 수 있다.
- [ ] `encoding='utf-8-sig'` 가 필요한 이유를 설명할 수 있다.
- [ ] 심화 도전 과제를 직접 구현해 보았다.

**다음 수업 연계**: 이 CSV를 입력 데이터로 사용하여  
머신러닝 흥행 예측 모델(회귀/분류)을 구축하는 실습으로 이어집니다.
